# 22 - Sensitivity analysis 4: lagged exposure (individual drugs)

Re-runs the primary individual-drug analysis and then the lagged-exposure analyses using `lag_sensitivity.py`.

**Manuscript items:** Table S17a

**Inputs:** derived cohort files from 01, comorbidity table from 02, CPRD DrugIssue extract, product dictionary, dosage lookup, interacting-drug code lists

**Outputs:** `SENS_LAG_DIR/lag_sensitivity_drug_name_*.xlsx` (one file per outcome and lag, plus retention table)

*Derived from the original analysis notebook `sensitivity_analysis/lag_period/hr_drugs.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
# Cap BLAS/OpenMP threads BEFORE numpy is imported. Without this, each of the
# worker processes below spawns its own thread pool and the machine thrashes.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter
import zipfile


In [ ]:
## show python, numpy, pandas, and lifelines versions
import sys
print(f"Python version: {sys.version}")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"))

In [ ]:
major_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), dtype=str)
major_doac_exposure["age_at_study_start"] = major_doac_exposure["age_at_study_start"].astype(int)
major_doac_exposure["study_start_date"] = pd.to_datetime(major_doac_exposure["study_start_date"])
major_doac_exposure["study_end_date"] = pd.to_datetime(major_doac_exposure["study_end_date"])
major_doac_exposure["event_date"] = pd.to_datetime(major_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

major_doac_exposure = major_doac_exposure[(major_doac_exposure["age_at_study_start"] >= 18) &
                (major_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (major_doac_exposure["taken_warfarin_1w_before"] =="0") &
                 (major_doac_exposure["taken_warfarin_during_study"] =="0") ]

major_doac_exposure = major_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date"]]
print(major_doac_exposure.shape[0])

In [ ]:
med_dict  = pd.read_csv(os.path.join(LOOKUP_DIR, "med_dict.txt"), sep="\t")

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)
prod_dict = prod_dict[prod_dict["DrugSubstanceName"].str.contains("/", case=False, na=False) == False]

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["doac_start_date"] = pats_data["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
pats_data["doac_start_date"] = pd.to_datetime(pats_data["doac_start_date"], errors="coerce")
pats_data = pats_data[pats_data["patid"].isin(major_doac_exposure["patid"])]
pats_data["age_at_study_start"] = (pats_data["doac_start_date"] - pd.to_datetime(pats_data["yob"], format="%Y")).dt.days // 365.25
## add the comorbidities to major_doac_exposure
#major_doac_exposure["age_at_study_start"] = major_doac_exposure["patid"].map(pats_data.set_index("patid")["age_at_study_start"])

In [ ]:
a = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_DrugInteractions_Table.xlsx"), dtype=str)
a = a[["Drug", "Category", "Action", "PK/PD"]]
drugs_action_increase = a[a["Action"] == "Increased"]
drugs_action_decrease = a[a["Action"] == "Decreased"]
p_k = drugs_action_increase[drugs_action_increase["PK/PD"] == "PK"]
p_d = drugs_action_increase[drugs_action_increase["PK/PD"] == "PD"]
ppi = a[(a["Category"]=="PPI")]

In [ ]:
drugs_action_increase[drugs_action_increase["Drug"]=="Enoxaparin"]

In [ ]:
interacting_drugs = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_interacting_drugs.xlsx"), dtype=str)
interacting_drugs = interacting_drugs[interacting_drugs["DrugSubstanceName"].str.contains("/", case=False, na=False) == False]
interacting_drugs["DrugSubstanceName"] = np.where(interacting_drugs["DrugSubstanceName"].isna(), interacting_drugs["matched_drugs"], interacting_drugs["DrugSubstanceName"])

interacting_drugs = interacting_drugs[
    interacting_drugs.apply(
        lambda row: row["DrugSubstanceName"].startswith(row["matched_drugs"][:3])
        if pd.notna(row["DrugSubstanceName"]) else False,
        axis=1
    )
]
words_to_remove = ["eye drops", "inhaler", "cream", "ointment", "gel", "spray", "lotion", "shampoo", "cream"]
interacting_drugs = interacting_drugs[interacting_drugs["Term from EMIS"].str.contains("|".join(words_to_remove), case=False, na=False) == False]
interacting_drugs_codes = interacting_drugs[["ProdCodeId", "matched_drugs"]]
interacting_drugs_codes.rename(columns={"matched_drugs": "drug_name", "ProdCodeId":"prodcodeid"}, inplace=True)
interacting_drugs_codes.drop_duplicates(subset=["prodcodeid"], inplace=True)
interacting_drugs_codes["drug_class"] = interacting_drugs_codes["drug_name"].map(a.set_index("Drug")["Category"])


In [ ]:
def calculate_enddate(df, dosage):
    df = df.copy()
    df["daily_dose"] = df["dosageid"].map(dosage.set_index("dosageid")["daily_dose"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["daily_dose"] = pd.to_numeric(df["daily_dose"], errors="coerce").fillna(0)

    # duration==0 rows that CAN be resolved from quantity / daily dose
    mask1 = (
        (df["duration"] == 0)
        & df["quantity"].notna()
        & (df["daily_dose"] != 0)
        & (df["quantity"] != 0)
    )
    # All other duration==0 rows (daily_dose unknown/zero, OR quantity missing/zero)
    # are imputed from the per-product median below, rather than being silently
    # left at 0 (which would otherwise produce a same-day enddate for an
    # unknown/unresolvable duration).
    mask2 = (df["duration"] == 0) & (~mask1)

    df["new_duration"] = np.where(mask1, df["quantity"] / df["daily_dose"], df["duration"])
    df["new_duration"] = pd.to_numeric(df["new_duration"], errors="coerce")

    # Median computed only from rows with a genuinely recorded (non-zero) duration,
    # so neither mask1-resolved rows nor other zero-duration rows distort it.
    median_durations = (
        df.loc[df["duration"] != 0].groupby("prodcodeid")["duration"].median()
    )

    df["new_duration"] = np.where(mask2, df["prodcodeid"].map(median_durations), df["new_duration"])

    df["enddate"] = (pd.to_datetime(df["issuedate"]) + pd.to_timedelta(df["new_duration"], unit="D"))

    return df

In [ ]:
dosage = pd.read_csv(os.path.join(LOOKUP_DIR, "common_dosages.txt"), sep="\t",dtype=str)


zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(interacting_drugs_codes["prodcodeid"])) & (df['patid'].isin(major_doac_exposure["patid"].tolist()))]

    df["study_start_date"] = df["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
    df["study_end_date"] = pd.to_datetime("2022-31-12")
    
    df = calculate_enddate(df, dosage)
    df = df[(df["issuedate"] < df["study_end_date"]) &(df["enddate"] > df["study_start_date"])]

    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Combine results
    interacting_drugs_exposure = pd.concat(all_results, ignore_index=True)

interacting_drugs_exposure["enddate"] = pd.to_datetime(interacting_drugs_exposure["enddate"], errors='coerce')
interacting_drugs_exposure["enddate"] = interacting_drugs_exposure["enddate"] + pd.to_timedelta(1, unit='D')
interacting_drugs_exposure.drop(columns=["dosageid", "duration", "quantity", "daily_dose", "probobsid", "new_duration"], inplace=True)
interacting_drugs_exposure.sort_values(['patid', 'issuedate'], inplace=True)
interacting_drugs_exposure.reset_index(drop=True, inplace=True)
interacting_drugs_exposure["drug_name"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_name"])

interacting_drugs_exposure["enddate"] = np.where(interacting_drugs_exposure["enddate"] > interacting_drugs_exposure["study_end_date"],
                                                  interacting_drugs_exposure["study_end_date"], interacting_drugs_exposure["enddate"])
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_name"].isin(drugs_action_increase["Drug"].tolist())]
#interacting_drugs_exposure["drug_name"] = np.where(interacting_drugs_exposure["drug_name"].isin(ppi["Drug"]), "PPI", interacting_drugs_exposure["drug_name"])


In [ ]:
major_interacting_drugs_presc = interacting_drugs_exposure.copy()
major_interacting_drugs_presc["study_end_date"] = major_interacting_drugs_presc["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])
major_interacting_drugs_presc = major_interacting_drugs_presc[(major_interacting_drugs_presc["issuedate"] < major_interacting_drugs_presc["study_end_date"]) ]
major_interacting_drugs_presc["enddate"] = np.where(major_interacting_drugs_presc["enddate"] > major_interacting_drugs_presc["study_end_date"], 
                                                    major_interacting_drugs_presc["study_end_date"], major_interacting_drugs_presc["enddate"])
major_interacting_drugs_presc.reset_index(drop=True, inplace=True)
major_interacting_drugs_presc["drug_term"] = major_interacting_drugs_presc["prodcodeid"].map(prod_dict.set_index("ProdCodeId")["Term from EMIS"])
major_interacting_drugs_presc.head(1)

In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"), dtype=str)
bleeding_codes

In [ ]:
major_events_df = major_doac_exposure[major_doac_exposure["event_date"].notnull()]
major_events_df

In [ ]:
presc = major_interacting_drugs_presc[["patid", "issuedate", "enddate", "drug_name"]]
events = major_events_df.reset_index(names="event_row")
merged = events.merge(presc,on="patid",how="left")

active = merged[(merged["issuedate"] <= merged["event_date"]) & (merged["enddate"] >= merged["event_date"])]
active_drugs = (active.groupby("event_row")["drug_name"].agg(list))

major_events_df["active_drugs_ls"] = (major_events_df.index.map(active_drugs))

major_events_df["active_drugs_ls"] = (major_events_df["active_drugs_ls"].apply(lambda x: x if isinstance(x, list) else []))

In [ ]:
from collections import defaultdict

drug_patients = defaultdict(set)
for _, row in major_events_df.iterrows():
    for drug in row["active_drugs_ls"]:
        drug_patients[drug].add(row["patid"])

drug_counts_df = pd.DataFrame([
    {"drug_name": drug, "count": len(patients)}
    for drug, patients in drug_patients.items()
])
drug_counts_df["presc_count"] = drug_counts_df["drug_name"].map(
    major_interacting_drugs_presc.groupby("drug_name")["patid"].nunique())
drug_counts_df["presc_count"] = drug_counts_df["presc_count"].fillna(0).astype(int)
drug_counts_df = drug_counts_df[(drug_counts_df["count"]>=10) ]# &
drug_counts_df = drug_counts_df.sort_values("count", ascending=False).reset_index(drop=True)
drug_counts_df = drug_counts_df[drug_counts_df["drug_name"]!="Enoxaparin"]
drug_counts_df

In [ ]:
drugs_to_consider  = drug_counts_df["drug_name"].tolist()
 
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_name"].isin(drugs_to_consider)].reset_index(drop=True)
interacting_drugs_exposure["drug_class"] = interacting_drugs_exposure["drug_name"].map(a.set_index("Drug")["Category"])


major_interacting_drugs_presc = major_interacting_drugs_presc[major_interacting_drugs_presc["drug_name"].isin(drugs_to_consider)].reset_index(drop=True)
major_interacting_drugs_presc["drug_class"] = major_interacting_drugs_presc["drug_name"].map(a.set_index("Drug")["Category"])

In [ ]:
def process_group(group):
    group = group.sort_values('issuedate').reset_index(drop=True)
    merged = []

    # Set days_gap based on drug_class
    drug_class = group['drug_class'].iloc[0]
    if drug_class == 'Antibiotic':
        days_gap = 7
    elif drug_class == 'Antifungal':
        days_gap = 7
    else:
        days_gap = 14  # default

    current_start = group.loc[0, 'issuedate']
    current_end = group.loc[0, 'enddate']

    for i in range(1, len(group)):
        next_start = group.loc[i, 'issuedate']
        next_end = group.loc[i, 'enddate']

        if (next_start - current_end).days < days_gap:
            current_end = max(current_end, next_end)
        else:
            merged.append({'patid': group.loc[i-1, 'patid'],
                           'drug_name': group.loc[i-1, 'drug_name'],
                           'drug_class': group.loc[i-1, 'drug_class'],
                           'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                           'enddate': min(current_end, group['study_end_date'].iloc[0])})
            current_start = next_start
            current_end = next_end

    merged.append({'patid': group.loc[len(group)-1, 'patid'],
                   'drug_name': group.loc[len(group)-1, 'drug_name'],
                   'drug_class': group.loc[len(group)-1, 'drug_class'],
                   'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                   'enddate': min(current_end, group['study_end_date'].iloc[0])})
    return pd.DataFrame(merged)

def merge_prescriptions(df):
    # Ensure dates are datetime
    for col in ['issuedate', 'enddate', 'study_start_date', 'study_end_date']:
        df[col] = pd.to_datetime(df[col])

    # Split groups
    groups = [group for _, group in df.groupby(['patid', 'drug_name'])]

    # Multiprocessing
    with Pool(60) as pool:
        result_list = pool.map(process_group, groups)

    merged_df = pd.concat(result_list, ignore_index=True)
    return merged_df

In [ ]:
major_interacting_drugs_presc_merged = merge_prescriptions(major_interacting_drugs_presc)
major_interacting_drugs_presc_merged["study_start_date"] = major_interacting_drugs_presc_merged["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
major_interacting_drugs_presc_merged["study_end_date"] = major_interacting_drugs_presc_merged["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

major_interacting_drugs_presc_merged

In [ ]:
def _process_patient(args):
    patid, group = args
    
    doac_start = group["study_start_date"].iloc[0]
    doac_end = group["study_end_date"].iloc[0]

    # clip exposures to DOAC period
    group = group.copy()
    group["start"] = group["issuedate"].clip(lower=doac_start, upper=doac_end)
    group["stop"] = group["enddate"].clip(lower=doac_start, upper=doac_end)

    # collect cut points
    cut_points = sorted(set([doac_start, doac_end]) | set(group["start"]) | set(group["stop"]))
    intervals = [(cut_points[i], cut_points[i+1]) for i in range(len(cut_points)-1)]

    rows = []
    for start, stop in intervals:
        if start >= stop:
            continue
        active = group[(group["start"] <= start) & (group["stop"] >= stop)]
        active_drugs = sorted(active["drug_name"].unique().tolist())
        rows.append({
            "patid": patid,
            "start_date": start,
            "end_date": stop,
            "concomitant_drugs_ls": active_drugs
        })
    return rows

def build_exposure_timeline(df, n_jobs=None):
    df = df.copy()
    df['issuedate'] = pd.to_datetime(df['issuedate'])
    df['enddate'] = pd.to_datetime(df['enddate'])
    df['study_start_date'] = pd.to_datetime(df['study_start_date'])
    df['study_end_date'] = pd.to_datetime(df['study_end_date'])
    
    grouped = list(df.groupby("patid"))

    with mp.Pool(processes=50) as pool:
        results = pool.map(_process_patient, grouped)

    # flatten list of lists
    flat_results = [row for sublist in results for row in sublist]
    return pd.DataFrame(flat_results)

In [ ]:
major_exposure_timeline_df = build_exposure_timeline(major_interacting_drugs_presc_merged)
major_exposure_timeline_df["study_start_date"] = major_exposure_timeline_df["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
major_exposure_timeline_df["study_end_date"] = major_exposure_timeline_df["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])

major_exposure_timeline_df["event_date"] = major_exposure_timeline_df["patid"].map(major_events_df.set_index("patid")["event_date"])
major_exposure_timeline_df["event_flag"] = np.where(
    (major_exposure_timeline_df["event_date"] >= major_exposure_timeline_df["start_date"]) & 
    (major_exposure_timeline_df["event_date"] <= major_exposure_timeline_df["end_date"]), 1, 0)


In [ ]:
pats_with_exposures = set(major_exposure_timeline_df["patid"].unique())

unique_major_events_df = major_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = major_doac_exposure[~major_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_major_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

major_exposure_timeline_df = pd.concat([major_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
major_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:
major_df = major_exposure_timeline_df.copy()
major_df = major_df[major_df["start_date"] < major_df["end_date"]].reset_index(drop=True)


for drug in drugs_to_consider:
    major_df[f"{drug}"] = major_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)
    
print(major_df[major_df["event_flag"]==1]["patid"].nunique())

In [ ]:
# Baseline PPI exposure: a PPI prescription in the 180 days before the index
# date. Enters the models only via the PPI-adjusted model (see below).
ppi_drug_codes = prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]
ppi_drug_codes = ppi_drug_codes[~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)]["ProdCodeId"].unique().tolist()

zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Ensure datetime conversion early
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')

    # Filter rows first (KEEP dataframe)
    df = df[
        df['prodcodeid'].isin(ppi_drug_codes) &
        df['patid'].isin(major_doac_exposure["patid"])
    ].copy()

    # Map study start date
    df["study_start_date"] = df["patid"].map(
        major_doac_exposure.set_index("patid")["study_start_date"]
    )

    df["study_start_date"] = pd.to_datetime(df["study_start_date"], errors='coerce')

    # Apply time window filter
    df = df[
        (df["issuedate"] >= df["study_start_date"] - pd.Timedelta(days=180)) &
        (df["issuedate"] < df["study_start_date"])
    ]

    return df["patid"].unique().tolist()


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    ppi_pats = set(p for sublist in all_results for p in sublist)

In [ ]:
# ===========================================================================
# Baseline covariates - RAW values, missing continuous values left as NaN
# ===========================================================================
# This cell no longer reads a pre-imputed workbook. It loads the raw
# comorbidity table so that the missing values in the continuous covariates are
# still present and can be handled by the multiple imputation performed further
# down (see "Multiple imputation" section). Imputation has to happen after the
# analysis frames exist, because the imputation model needs the outcome and the
# Nelson-Aalen cumulative hazard (White & Royston 2009).

comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype={"patid": str})
comorbidities = comorbidities[comorbidities["patid"].isin(major_doac_exposure["patid"])].copy()

binary_comorbidity_cols = ["major_bled_hist", "hypertension", "liver_disease", "stroke", "gender", "heart_failure",
                           "atrial_fibrillation", "venous_thromboembolism", "cancer", "diabetes", "smoking", "alcohol"]
continuous_comorbidity_cols = ["age_at_study_start", "ckd_value", "bmi_value"]

comorbidities["gender"] = comorbidities["patid"].map(pats_data.set_index("patid")["gender"])

# Continuous covariates are coerced to numeric so that blanks / sentinel strings
# become NaN rather than silently surviving as object dtype.
for col in continuous_comorbidity_cols:
    comorbidities[col] = pd.to_numeric(comorbidities[col], errors="coerce")

# ---- Missingness report (quote these numbers in the methods) ---------------
_n_pat = comorbidities["patid"].nunique()
missing_report = pd.DataFrame([
    {"variable": c,
     "n_missing": int(comorbidities[c].isna().sum()),
     "pct_missing": round(100 * comorbidities[c].isna().mean(), 2)}
    for c in continuous_comorbidity_cols
])
print(f"Patients in comorbidity table: {_n_pat:,}")
print(missing_report.to_string(index=False))

for col in binary_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

# NOTE: NaNs are deliberately carried through here - they are filled by the
# imputation step, and the complete-case sensitivity analysis needs to know
# which patients were originally missing.
for col in continuous_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col])

major_df["PPI_baseline"] = major_df["patid"].isin(ppi_pats).astype(int)


In [ ]:
# ===========================================================================
# Two-model design: PPI adjustment applied ONLY to NSAIDs and aspirin
# ===========================================================================
# A single covariate in one joint model necessarily adjusts every drug's hazard
# ratio. To adjust only NSAIDs and aspirin for baseline PPI use, the same fully
# mutually adjusted model is therefore fitted twice:
#
#   fits["no_ppi"]    all drug terms + comorbidities                  (no PPI)
#   fits["with_ppi"]  all drug terms + comorbidities + PPI_baseline
#
# The reported table takes the NSAID and aspirin rows from the PPI-adjusted fit
# and every other drug from the unadjusted fit. Both fits retain the full set of
# interacting-drug terms, so mutual adjustment between drugs is preserved in
# both. A "PPI adjusted" column records which model each row came from - state
# this in the methods, because the published table then mixes estimates from two
# models rather than reading off a single one.

# ---- Which drugs get the PPI adjustment ----------------------------------
# Read from the Category column of DOAC_DrugInteractions_Table.xlsx (dataframe
# `a`) rather than hard coded, then intersected with the drugs that cleared the
# >=10 serious bleeds threshold.
_is_nsaid = _is_nsaid = a["Category"].isin(["NSAID"])

_ppi_adjust_lookup = (
    set(a.loc[_is_nsaid, "Drug"].dropna().str.strip().str.lower()) | {"aspirin"})
ppi_adjusted_drugs = sorted(
    d for d in drugs_to_consider if str(d).strip().lower() in _ppi_adjust_lookup)

print("Drugs adjusted for baseline PPI use:", ppi_adjusted_drugs)
if not ppi_adjusted_drugs:
    raise ValueError(
        "No NSAID or aspirin exposures identified - check the 'Category' labels "
        "in DOAC_DrugInteractions_Table.xlsx before fitting.")

# ---- Formulas and the columns each fit needs ------------------------------
drug_terms = list(drugs_to_consider)
comorbidity_terms = binary_comorbidity_cols + continuous_comorbidity_cols
_index_cols = ["patid", "start_date", "end_date", "event_flag"]

covariates_no_ppi = " + ".join(drug_terms + comorbidity_terms)
covariates_with_ppi = covariates_no_ppi + " + PPI_baseline"

MODEL_SPECS = {
    "no_ppi": (covariates_no_ppi, _index_cols + drug_terms + comorbidity_terms),
    "with_ppi": (covariates_with_ppi,
                 _index_cols + drug_terms + comorbidity_terms + ["PPI_baseline"]),
}

# Kept for backwards compatibility with any cell still referencing `covariates`.
covariates = covariates_no_ppi


In [ ]:
# ---------- Helper functions (defined once, before any use) ----------
from scipy import stats

def calculate_crude_hr(df, drug_class):
    ctv_crude = CoxTimeVaryingFitter()
    ctv_crude.fit(
        df,
        id_col='patid',
        start_col='start_date',
        stop_col='end_date',
        event_col='event_flag',
        formula=drug_class,
        show_progress=True
    )
    row = ctv_crude.summary.loc[drug_class]
    return row["exp(coef)"], row["exp(coef) lower 95%"], row["exp(coef) upper 95%"], row["p"]


def _as_list(x):
    """concomitant_drugs_ls holds python lists, but NaN can survive a concat."""
    return x if isinstance(x, (list, tuple, set)) else []


def reference_mask(df, all_terms=None):
    """Boolean mask over interval rows: person-time with NO interacting drug active.

    This is the reference level the mutually adjusted Cox models actually
    contrast against - every interacting-drug indicator is 0 - and so it is the
    only comparator that makes an absolute measure consistent with the hazard
    ratio sitting next to it in the table.

    `df[drug] == 0` is NOT this comparator. It also contains person-time exposed
    to the *other* interacting drugs, which carries a higher event rate. Using it
    is what made the old comparator rate come out ABOVE the cohort-wide rate
    (~1.87 vs 1.8 per 100 PY), which is arithmetically impossible for a genuine
    unexposed stratum when every exposed stratum sits above the cohort mean.

    Derived from `concomitant_drugs_ls` where available so that the mask does not
    depend on which subset of terms is being tabulated (build_split_ppi_summary
    tabulates the NSAID/aspirin rows separately from the rest, and both must
    share one reference).
    """
    if "concomitant_drugs_ls" in df.columns:
        return df["concomitant_drugs_ls"].map(
            lambda v: len(_as_list(v)) == 0).to_numpy(dtype=bool)
    if all_terms:
        cols = [t for t in all_terms if t in df.columns]
        if not cols:
            raise ValueError("None of `all_terms` are columns of df.")
        return (df[cols].sum(axis=1) == 0).to_numpy(dtype=bool)
    raise ValueError(
        "Cannot identify unexposed person-time: pass `all_terms`, or keep the "
        "`concomitant_drugs_ls` column on the analysis frame.")


def _events_and_py(df, mask):
    """Events (unique patients) and person-years over a set of interval rows."""
    sub = df.loc[mask]
    if len(sub) == 0:
        return 0, 0.0, np.nan
    # total_seconds rather than .dt.days: .days truncates every interval towards
    # zero, which silently shortens person-time across millions of rows.
    days = ((sub["end_date"] - sub["start_date"]).dt.total_seconds() / 86400.0).clip(lower=0)
    py = float(days.sum()) / 365.25
    events = int(sub.loc[sub["event_flag"] == 1, "patid"].nunique())
    rate100 = (events / py) * 100 if py > 0 else np.nan
    return events, py, rate100


def calculate_events_person_years(df, drug_class, exposed=True):
    """Crude events / person-years / rate per 100 PY for one term.

    Kept for the descriptive 'Events/Person Years' and 'Events per 100 PY'
    columns. `exposed=False` means 'not on this drug' and is NOT the model
    reference - use reference_mask() for anything compared against a hazard
    ratio.
    """
    flag = 1 if exposed else 0
    return _events_and_py(df, (df[drug_class] == flag).to_numpy(dtype=bool))


def _se_and_crit_from_row(row):
    """Standard error of log HR and the two-sided 95% critical value.

    Uses the model's own SE. After Rubin pooling the interval is built on a
    Barnard-Rubin t quantile rather than 1.96, so the same quantile is reused
    here; otherwise the rate-difference CI would not be on the same footing as
    the HR CI printed beside it.
    """
    se = float(row["se(coef)"]) if "se(coef)" in row.index and pd.notna(row["se(coef)"]) else np.nan
    if not np.isfinite(se):
        lo, hi = float(row["exp(coef) lower 95%"]), float(row["exp(coef) upper 95%"])
        se = (np.log(hi) - np.log(lo)) / (2 * 1.959963985)
    dof = float(row["df"]) if "df" in row.index and pd.notna(row["df"]) else np.inf
    crit = float(stats.t.ppf(0.975, dof)) if np.isfinite(dof) else 1.959963985
    return se, crit


def calculate_nnh(df, drug_class, model_row, all_terms=None, reference=None, scale=1000):
    """Adjusted rate difference and person-years-per-additional-event.

    The absolute measure is now derived from the adjusted hazard ratio rather
    than from two crude rates:

        RD = r0 * (HR_adj - 1)

    where r0 is the event rate in unexposed person-time (no interacting drug
    active) and HR_adj is the mutually adjusted, MI-pooled hazard ratio for this
    drug. Under the proportional hazards model the adjusted rate in exposed time
    is r0 * HR_adj, so RD is the excess rate attributable to the exposure with
    the confounders held fixed. The previous version subtracted two crude rates
    and never touched the model at all, so the number labelled "adjusted rate
    difference" in the manuscript was not adjusted for anything.

    CI by the delta method on RD = exp(a) * (exp(b) - 1), a = log r0, b = log HR:

        Var(RD) = r0^2 * [ (HR - 1)^2 * Var(log r0) + HR^2 * Var(log HR) ]

    with Var(log r0) = 1 / events_ref (Poisson). r0 and HR_adj are treated as
    independent; they are weakly correlated in truth (the reference person-time
    contributes to the partial likelihood), but the correlation is negligible
    relative to the two variance terms and ignoring it is conservative here.

    NNH is returned in PERSON-YEARS of exposure per additional event. It is not a
    number of persons - report it under that name in the tables and text.
    """
    ref = reference_mask(df, all_terms) if reference is None else reference

    events_ref, py_ref, rate_ref_100 = _events_and_py(df, ref)
    events_exp, py_exp, rate_exp_100 = calculate_events_person_years(df, drug_class, exposed=True)

    hr = float(model_row["exp(coef)"])
    se_log_hr, crit = _se_and_crit_from_row(model_row)

    out = {
        "ref_events": events_ref,
        "ref_person_years": round(py_ref, 2),
        "rate_unexposed_per_100py": round(rate_ref_100, 3) if np.isfinite(rate_ref_100) else np.nan,
        "HR_used_for_RD": round(hr, 4),
    }

    if events_ref == 0 or py_ref <= 0 or not np.isfinite(hr):
        nan_cols = {
            f"rate_diff_per_{scale}py": np.nan,
            f"rate_diff_per_{scale}py_lower": np.nan,
            f"rate_diff_per_{scale}py_upper": np.nan,
            "NNH": "Not estimable", "NNH_CI_lower": np.nan, "NNH_CI_upper": np.nan,
            "CI_crosses_null": True,
        }
        return {**out, **nan_cols}

    r0 = rate_ref_100 / 100.0                      # events per person-year
    rd = r0 * (hr - 1.0)                           # adjusted rate difference

    var_log_r0 = 1.0 / events_ref
    var_rd = (r0 ** 2) * ((hr - 1.0) ** 2 * var_log_r0 + (hr ** 2) * se_log_hr ** 2)
    se_rd = np.sqrt(var_rd)
    rd_lower, rd_upper = rd - crit * se_rd, rd + crit * se_rd

    rd_s, rd_lo_s, rd_hi_s = rd * scale, rd_lower * scale, rd_upper * scale

    def _nnh(x):
        # x is a rate difference on the `scale` person-year basis; scale / x is
        # therefore person-years per event regardless of the scale chosen.
        return scale / x if np.isfinite(x) and x > 0 else np.nan

    nnh = _nnh(rd_s)
    nnh_lower = _nnh(rd_hi_s)      # bounds flip under the reciprocal
    nnh_upper = _nnh(rd_lo_s)
    crosses_null = (not np.isfinite(rd_lo_s)) or (rd_lo_s <= 0 <= rd_hi_s)

    out.update({
        f"rate_diff_per_{scale}py": round(rd_s, 3),
        f"rate_diff_per_{scale}py_lower": round(rd_lo_s, 3),
        f"rate_diff_per_{scale}py_upper": round(rd_hi_s, 3),
        # "Not estimable" when HR <= 1: no excess rate, so the reciprocal is
        # undefined. NaN in the upper bound means the RD interval crossed zero
        # and the bound runs to infinity.
        "NNH": round(nnh, 1) if np.isfinite(nnh) else "Not estimable",
        "NNH_CI_lower": round(nnh_lower, 1) if np.isfinite(nnh_lower) else np.nan,
        "NNH_CI_upper": round(nnh_upper, 1) if np.isfinite(nnh_upper) else np.nan,
        "CI_crosses_null": crosses_null,
    })
    return out


def calculate_crude_rate_difference(df, drug_class, scale=1000):
    """The old, unadjusted contrast, kept only so the two can be reported
    side by side. Comparator here is 'not on this drug', which still includes
    time on other interacting drugs - it is descriptive, not a causal contrast.
    """
    _, _, rate_exp_100 = calculate_events_person_years(df, drug_class, exposed=True)
    _, _, rate_unexp_100 = calculate_events_person_years(df, drug_class, exposed=False)
    rd = (rate_exp_100 - rate_unexp_100) / 100.0
    return {f"crude_rate_diff_per_{scale}py": round(rd * scale, 3),
            "crude_NNH": round(1.0 / rd, 1) if rd > 0 else "Not estimable"}


def build_bleeding_summary(ctv_model, df, drug_classes, nnh_scale=1000,
                           all_terms=None, include_crude_rd=True):
    # Accepts either a fitted CoxTimeVaryingFitter or a `.summary` DataFrame
    # (the parallel fitters return the latter, since a whole fitted model is
    # expensive to send back from a worker process).
    _raw = ctv_model if isinstance(ctv_model, pd.DataFrame) else ctv_model.summary
    summary = _raw.reset_index().rename(columns={
        "covariate": "drug_class", "exp(coef)": "HR",
        "exp(coef) lower 95%": "CI Lower", "exp(coef) upper 95%": "CI Upper",
    })
    summary = summary[summary["drug_class"].isin(drug_classes)].copy()

    summary["p"] = summary["p"].round(4)
    for col in ["HR", "CI Lower", "CI Upper"]:
        summary[col] = summary[col].round(2)
    summary = summary.sort_values("HR", ascending=False).reset_index(drop=True)

    # One reference stratum for the whole table, computed once. Every hazard
    # ratio in the model is a contrast against this same drug-free person-time,
    # so every absolute measure must use it too - including the rows that
    # build_split_ppi_summary tabulates from the PPI-adjusted fit.
    ref = reference_mask(df, all_terms)
    ref_events, ref_py, ref_rate = _events_and_py(df, ref)
    print(f"    reference (no interacting drug active): {ref_events:,} events / "
          f"{ref_py:,.0f} person-years = {ref_rate:.3f} per 100 PY")

    person_time_rows = []
    for drug in drug_classes:
        events, py, rate = calculate_events_person_years(df, drug, exposed=True)
        nnh_res = calculate_nnh(df, drug, _raw.loc[drug], reference=ref, scale=nnh_scale)
        row = {
            "drug_class": drug,
            "no_of_events": events,
            "person_years": round(py, 2),
            "events_per_100py": round(rate, 2),
            **nnh_res,
        }
        if include_crude_rd:
            row.update(calculate_crude_rate_difference(df, drug, scale=nnh_scale))
        person_time_rows.append(row)
    person_time_df = pd.DataFrame(person_time_rows)

    summary = summary.merge(person_time_df, on="drug_class", how="left")
    summary["person_years"] = summary["person_years"].fillna(0).astype(int)
    summary["events/person_years"] = summary["no_of_events"].astype(str) + "/" + summary["person_years"].astype(str)
    summary["Patient Counts"] = summary["drug_class"].map(lambda d: df[df[d] == 1]["patid"].nunique())
    summary["No of Events"] = summary["no_of_events"]

    nnh_cols = [c for c in person_time_df.columns
                if c not in ("drug_class", "no_of_events", "person_years", "events_per_100py")]

    return summary[[
        "drug_class", "Patient Counts", "events/person_years", "events_per_100py",
        "HR", "CI Lower", "CI Upper", "p", "No of Events",
        *nnh_cols,
    ]]


# ---------- Parallel Cox fitting ----------------------------------------------
# Every fit below is independent, so they are run in worker processes. The data
# frame is published to a module-level global BEFORE the Pool is created: on
# Linux the workers are forked and inherit it copy-on-write, so a large frame is
# never pickled or duplicated. This is the same pattern the DrugIssue readers
# further up the notebook already rely on, and it requires the default 'fork'
# start method (mp.get_start_method() should return 'fork').

CRUDE_N_JOBS = 20      # crude fits are small; raise if you have the cores
ADJUSTED_N_JOBS = 2    # one worker per adjusted model

_FIT_DF = None         # inherited by forked workers, never pickled


def _fit_adjusted_worker(args):
    label, formula, cols = args
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        _FIT_DF[cols],
        id_col='patid', start_col='start_date', stop_col='end_date',
        event_col='event_flag', formula=formula, show_progress=False,
    )
    return label, ctv.summary


def _fit_crude_worker(term):
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        _FIT_DF[["patid", "start_date", "end_date", "event_flag", term]],
        id_col='patid', start_col='start_date', stop_col='end_date',
        event_col='event_flag', formula=term, show_progress=False,
    )
    row = ctv.summary.loc[term]
    return {
        "drug_class": term,
        "Crude HR": round(row["exp(coef)"], 2),
        "Crude CI Lower": round(row["exp(coef) lower 95%"], 2),
        "Crude CI Upper": round(row["exp(coef) upper 95%"], 2),
        "Crude p": round(row["p"], 4),
    }


def _run_pool(df, worker, tasks, n_jobs):
    global _FIT_DF
    _FIT_DF = df
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            return pool.map(worker, tasks)
    finally:
        _FIT_DF = None


def fit_adjusted_models(df, specs=None, n_jobs=ADJUSTED_N_JOBS, label=""):
    """Fit the no-PPI and with-PPI models in parallel. Returns {label: summary}."""
    specs = MODEL_SPECS if specs is None else specs
    t0 = time.time()
    tasks = [(k, f, c) for k, (f, c) in specs.items()]
    out = dict(_run_pool(df, _fit_adjusted_worker, tasks, n_jobs))
    print(f"[{label}] fitted {len(out)} adjusted models in {time.time()-t0:.1f}s")
    return out


def crude_hr_table(df, terms, n_jobs=CRUDE_N_JOBS, label=""):
    """Univariate HR for each term, one worker per term."""
    t0 = time.time()
    rows = _run_pool(df, _fit_crude_worker, list(terms), n_jobs)
    print(f"[{label}] fitted {len(rows)} crude models in {time.time()-t0:.1f}s")
    return pd.DataFrame(rows)


def build_split_ppi_summary(fits, df, drug_terms, ppi_adjusted, nnh_scale=1000):
    """NSAID/aspirin rows from the PPI-adjusted fit, all other drugs from the
    unadjusted fit, concatenated into one table."""
    ppi_set = set(ppi_adjusted)
    groups = [
        ("no_ppi", [d for d in drug_terms if d not in ppi_set], "No"),
        ("with_ppi", [d for d in drug_terms if d in ppi_set], "Yes"),
    ]
    parts = []
    for key, terms, flag in groups:
        if not terms:
            continue
        # all_terms=drug_terms: the reference stratum is drug-free person-time
        # for BOTH groups, not 'free of the terms in this group'.
        part = build_bleeding_summary(fits[key], df, terms, nnh_scale,
                                      all_terms=drug_terms)
        part.insert(1, "PPI adjusted", flag)
        parts.append(part)
    out = pd.concat(parts, ignore_index=True)
    return out.sort_values("HR", ascending=False).reset_index(drop=True)


## Multiple imputation of missing continuous covariates

Missing values in the continuous baseline covariates (`ckd_value` / eGFR and
`bmi_value`) are handled by **multiple imputation by chained equations (MICE)**
using `miceforest` (random forest / LightGBM chained equations with predictive
mean matching), rather than by complete-case analysis or a missing-indicator
category.

Three things make this survival-analysis-appropriate rather than a generic MICE run:

1. **The outcome is in the imputation model.** The event indicator, follow-up
   time and the **Nelson-Aalen cumulative hazard** estimate are included as
   predictors, following White IR & Royston P, *Stat Med* 2009;28:1982-1998.
   Omitting the outcome biases covariate-outcome associations toward the null.
2. **The imputation model is congenial with the substantive model.** Every
   covariate in the Cox models is a predictor in the imputation model, plus
   ever-exposed indicators for each interacting drug and baseline PPI use, so
   the imputation model is at least as rich as the analysis model.
3. **Estimates are pooled by Rubin's rules** on the log-hazard scale, with
   Barnard-Rubin degrees of freedom, and the fraction of missing information
   (FMI) is reported for each coefficient.

Imputation is run **separately for each outcome cohort** (serious, minor, all
bleeding), because the outcome and follow-up time entering the imputation model
differ between them.

A **complete-case sensitivity analysis** is run alongside, as recommended by
Sterne JAC et al., *BMJ* 2009;338:b2393 and required for STROBE / RECORD-PE
reporting.


In [ ]:
# ===========================================================================
# Multiple imputation helpers (MICE + Rubin's rules)
# ===========================================================================
from scipy import stats
from lifelines import NelsonAalenFitter

# ---- Configuration --------------------------------------------------------
# m = number of imputed datasets. Set to 10 to match the manuscript and the
# White/Royston/Wood (2011) rule of thumb that m should be at least the
# percentage of incomplete cases (BMI ~8.4% missing). Set to 5 if you prefer
# the smaller number - nothing else needs to change.
MI_CONFIG = dict(
    m=10,
    iterations=5,
    seed=20240824,
    mean_match_candidates=5,   # predictive mean matching: imputed values are
                               # drawn from observed values of near neighbours
    n_jobs=10,                 # imputed datasets are independent draws, so each
                               # one is built in its own worker process. Set to 1
                               # to fall back to a single m-dataset kernel.
)
IMPUTE_COLS = ["ckd_value", "bmi_value"]   # continuous covariates to impute

# The 2 x m adjusted fits per outcome are all independent, so they are farmed
# out as a single flat pool of tasks rather than looping over imputations. Each
# worker builds its own copy of the design frame, so peak memory is roughly
# MI_FIT_N_JOBS x (size of the interval-level frame restricted to model
# columns). Lower this first if the node runs out of memory.
MI_FIT_N_JOBS = 20

# NOTE ON THREADS: the first cell of this notebook pins OMP_NUM_THREADS=1, which
# also silences LightGBM's internal threading inside miceforest. That is the
# right default here because MICE is parallelised across processes instead.
# If you set MI_CONFIG["n_jobs"]=1, give LightGBM threads back explicitly:
#     run_mice(..., num_threads=16)      # **lgb_kwargs are forwarded to mice()
# Do not do both at once - the two layers will oversubscribe the cores.


def check_mi_versions():
    """miceforest 6.0.x is incompatible with lightgbm >= 4.6 (the
    Booster.__inner_predict signature changed). Fail loudly and early rather
    than 40 minutes into a run."""
    import miceforest, lightgbm
    print(f"miceforest {miceforest.__version__} | lightgbm {lightgbm.__version__}")
    major_minor = tuple(int(x) for x in lightgbm.__version__.split(".")[:2])
    if major_minor >= (4, 6):
        print("WARNING: kernel.mice() may raise "
              "'Booster.__inner_predict() takes 1 positional argument'. "
              "If it does, run:  pip install 'lightgbm<4.6'")


# ---- 1. Nelson-Aalen cumulative hazard (White & Royston 2009) --------------
def nelson_aalen_cumhaz(durations, events):
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    naf = NelsonAalenFitter()
    naf.fit(durations, event_observed=events)
    ch = naf.cumulative_hazard_.iloc[:, 0]
    times = ch.index.to_numpy(dtype=float)
    idx = np.clip(np.searchsorted(times, durations, side="right") - 1, 0, len(times) - 1)
    return ch.to_numpy()[idx]


# ---- 2. Patient-level frame for the imputation model ----------------------
def build_imputation_frame(analysis_df, comorbidities, binary_cols, continuous_cols,
                           drug_terms=(), extra_binary=("PPI_baseline",)):
    """One row per patient: total follow-up, event indicator, Nelson-Aalen
    cumulative hazard, every analysis covariate, and ever-exposed flags for the
    interacting drugs (so the imputation model is congenial with the Cox model)."""
    df = analysis_df
    terms = [t for t in list(drug_terms) + list(extra_binary) if t in df.columns]

    slim = pd.DataFrame({
        "patid": df["patid"].to_numpy(),
        "_dur": (df["end_date"] - df["start_date"]).dt.days.clip(lower=0).to_numpy(),
        "event_flag": df["event_flag"].to_numpy(),
    })
    for t in terms:
        slim[t] = df[t].to_numpy().astype("int8")

    agg_map = {"fu_days": ("_dur", "sum"), "event": ("event_flag", "max")}
    agg_map.update({f"ever_{t}": (t, "max") for t in terms})
    pat = slim.groupby("patid", sort=False).agg(**agg_map).reset_index()

    pat["fu_years"] = (pat.pop("fu_days") / 365.25).clip(lower=1 / 365.25)
    pat["event"] = pat["event"].astype(int)
    pat["na_cumhaz"] = nelson_aalen_cumhaz(pat["fu_years"], pat["event"])
    pat["log_fu_years"] = np.log(pat["fu_years"])
    for t in terms:
        pat[f"ever_{t}"] = pat[f"ever_{t}"].astype(float)

    com = comorbidities.drop_duplicates(subset=["patid"]).set_index("patid")
    for col in binary_cols:
        if col in com.columns:
            pat[col] = pd.to_numeric(pat["patid"].map(com[col]), errors="coerce").fillna(0).astype(float)
    for col in continuous_cols:
        if col in com.columns:
            pat[col] = pd.to_numeric(pat["patid"].map(com[col]), errors="coerce")

    return pat


# ---- 3. MICE with miceforest ----------------------------------------------
def _make_kernel(work, m, seed, mean_match_candidates=5):
    """Construct an ImputationKernel across miceforest 5.x / 6.x signatures."""
    import miceforest as mf
    attempts = [
        dict(num_datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations_data=True),
        dict(datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations=True),
        dict(num_datasets=m, random_state=seed),
        dict(datasets=m, random_state=seed),
    ]
    last = None
    for kw in attempts:
        try:
            return mf.ImputationKernel(work, **kw)
        except TypeError as exc:
            last = exc
    raise RuntimeError(f"Could not construct miceforest ImputationKernel: {last}")


_MICE_WORK = None       # published to forked workers, never pickled
_MICE_TARGETS = None


def _mice_worker(args):
    """One imputed dataset, built by its own single-dataset kernel."""
    idx, seed, iterations, mmc, lgb_kwargs = args
    kernel = _make_kernel(_MICE_WORK, 1, seed, mmc)
    kernel.mice(iterations, **lgb_kwargs)
    comp = kernel.complete_data(dataset=0)
    return idx, {c: np.asarray(comp[c], dtype=float) for c in _MICE_TARGETS}


def run_mice_parallel(pat_frame, continuous_cols, targets, m, iterations, seed,
                      mean_match_candidates, n_jobs, label="", **lgb_kwargs):
    """m independent chains in m worker processes.

    Statistically this is the same estimator as one m-dataset kernel: the m
    imputations are independent draws either way, the only difference being that
    each chain gets its own RNG stream. Streams are spawned from a single
    SeedSequence so the whole run is reproducible from MI_CONFIG["seed"]."""
    global _MICE_WORK, _MICE_TARGETS
    t0 = time.time()
    ss = np.random.SeedSequence(seed)
    seeds = [int(s.generate_state(1)[0]) % (2 ** 31 - 1) for s in ss.spawn(m)]
    tasks = [(i, seeds[i], iterations, mean_match_candidates, lgb_kwargs) for i in range(m)]

    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    _MICE_WORK, _MICE_TARGETS = work, targets
    try:
        with Pool(min(n_jobs, m)) as pool:
            results = dict(pool.map(_mice_worker, tasks))
    finally:
        _MICE_WORK = _MICE_TARGETS = None

    out = []
    for i in range(m):
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in results[i]:
                frame[c] = results[i][c]
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE: {m} datasets on {min(n_jobs, m)} workers "
          f"in {time.time() - t0:.1f}s | seeds {seeds[:3]}...")
    return out, seeds


def run_mice(pat_frame, continuous_cols, m=10, iterations=5, seed=20240824,
             mean_match_candidates=5, n_jobs=1, label="", verbose=True, **lgb_kwargs):
    """Returns (list of m patid->imputed-value frames, fitted kernel or seeds)."""
    t0 = time.time()
    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    target_cols = [c for c in work.columns if work[c].isna().any()]
    if verbose:
        print(f"[{label}] imputing {target_cols} | m={m}, iterations={iterations}, "
              f"predictors={work.shape[1] - 1}, n={len(work):,}")
        extra = [c for c in target_cols if c not in continuous_cols]
        if extra:
            print(f"[{label}] note: these predictors also had missing values and "
                  f"were imputed jointly: {extra}")
    if not target_cols:
        print(f"[{label}] nothing to impute - no missing values found.")
        base = pat_frame[["patid"] + [c for c in continuous_cols if c in pat_frame.columns]].copy()
        return [base.copy() for _ in range(m)], None

    if n_jobs and n_jobs > 1 and m > 1:
        return run_mice_parallel(pat_frame, continuous_cols, target_cols, m, iterations,
                                 seed, mean_match_candidates, n_jobs, label=label,
                                 **lgb_kwargs)

    kernel = _make_kernel(work, m, seed, mean_match_candidates)
    kernel.mice(iterations, **lgb_kwargs)

    out = []
    for i in range(m):
        comp = kernel.complete_data(dataset=i)
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in comp.columns:
                frame[c] = np.asarray(comp[c])
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE finished in {time.time() - t0:.1f}s")
    return out, kernel


def imputation_diagnostics(pat_frame, imputations, continuous_cols):
    """Observed vs imputed distributions - include in the supplement."""
    rows = []
    for c in continuous_cols:
        if c not in pat_frame.columns:
            continue
        miss = pat_frame[c].isna().to_numpy()
        obs = pat_frame.loc[~miss, c]
        rows.append(dict(variable=c, source="observed", n=int(obs.size), mean=obs.mean(),
                         sd=obs.std(), p25=obs.quantile(.25), median=obs.median(),
                         p75=obs.quantile(.75)))
        if miss.sum() == 0:
            continue
        for i, imp in enumerate(imputations, 1):
            v = pd.to_numeric(imp.loc[miss, c])
            rows.append(dict(variable=c, source=f"imputed {i}", n=int(v.size), mean=v.mean(),
                             sd=v.std(), p25=v.quantile(.25), median=v.median(),
                             p75=v.quantile(.75)))
    return pd.DataFrame(rows).round(3)


# ---- 4. Rubin's rules ------------------------------------------------------
def rubin_pool(summaries, n_events=None, n_params=None):
    """Pool lifelines `.summary` frames on the log-hazard scale.

    Returns a frame shaped like a lifelines summary (so it can be passed
    straight to build_bleeding_summary) plus FMI, RIV and Barnard-Rubin df."""
    coef = pd.concat([s["coef"] for s in summaries], axis=1)
    se = pd.concat([s["se(coef)"] for s in summaries], axis=1)
    m = coef.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")

    qbar = coef.mean(axis=1)                 # pooled point estimate
    ubar = (se ** 2).mean(axis=1)            # within-imputation variance
    b = coef.var(axis=1, ddof=1)             # between-imputation variance
    total = ubar + (1 + 1 / m) * b           # total variance
    se_pool = np.sqrt(total)

    lam = np.where(total > 0, ((1 + 1 / m) * b) / total, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(ubar > 0, ((1 + 1 / m) * b) / ubar, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)

    if n_events and n_params:                # Barnard & Rubin (1999) adjustment
        nu_com = max(int(n_events) - int(n_params), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    out = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "coef lower 95%": qbar - crit * se_pool,
        "coef upper 95%": qbar + crit * se_pool,
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "riv": r,
        "df": df_adj,
        "n_imputations": m,
    })
    out.index.name = "covariate"
    return out


# ---- 5. Drivers: fit across imputations, pool, and complete-case fallback --
_MI_BASE_DF = None      # inherited copy-on-write by forked workers
_MI_VALUES = None       # per-imputation, row-aligned arrays of imputed values


def _fit_mi_worker(args):
    imp_idx, key, formula, cols = args
    sub = _MI_BASE_DF[cols].copy()
    for c, arr in _MI_VALUES[imp_idx].items():
        if c in sub.columns:
            sub[c] = arr
    ctv = CoxTimeVaryingFitter()
    ctv.fit(sub, id_col="patid", start_col="start_date", stop_col="end_date",
            event_col="event_flag", formula=formula, show_progress=False)
    return imp_idx, key, ctv.summary


def fit_adjusted_models_mi(df, imputations, impute_cols=None, specs=None,
                           n_jobs=None, label=""):
    """Refit both adjusted models on every imputed dataset and pool.

    The 2 x m fits are independent, so they run as one flat pool of tasks. The
    imputed values are mapped onto the interval-level rows ONCE in the parent
    and inherited copy-on-write; each worker only pays for its own design frame.
    `df` itself is never modified."""
    impute_cols = IMPUTE_COLS if impute_cols is None else list(impute_cols)
    specs = MODEL_SPECS if specs is None else specs
    n_jobs = MI_FIT_N_JOBS if n_jobs is None else n_jobs
    global _MI_BASE_DF, _MI_VALUES
    t0 = time.time()

    _model_cols = sorted({c for _, cols in specs.values() for c in cols}
                         - {"patid", "start_date", "end_date", "event_flag"})

    values = []
    for i, imp in enumerate(imputations, 1):
        lut = imp.set_index("patid")
        vals = {c: df["patid"].map(lut[c]).to_numpy(dtype=float)
                for c in impute_cols if c in lut.columns}
        for c, arr in vals.items():
            n_bad = int(np.isnan(arr).sum())
            if n_bad:
                raise ValueError(f"{c}: {n_bad} rows still NaN after imputation {i} - "
                                 "some patids are absent from the imputation frame.")
        values.append(vals)

    _still = {c: int(df[c].isna().sum()) for c in _model_cols
              if c not in impute_cols and df[c].isna().any()}
    if _still:
        raise ValueError(f"Model covariates outside IMPUTE_COLS still contain NaN: {_still}. "
                         "Add them to IMPUTE_COLS so they are written back from the "
                         "imputed datasets.")

    tasks = [(i, k, f, c) for i in range(len(imputations)) for k, (f, c) in specs.items()]
    _MI_BASE_DF, _MI_VALUES = df, values
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            res = pool.map(_fit_mi_worker, tasks)
    finally:
        _MI_BASE_DF = _MI_VALUES = None

    per_model = {k: [None] * len(imputations) for k in specs}
    for imp_idx, key, summ in res:
        per_model[key][imp_idx] = summ

    n_events = int(df.loc[df["event_flag"] == 1, "patid"].nunique())
    pooled = {k: rubin_pool(v, n_events=n_events, n_params=len(v[0]))
              for k, v in per_model.items()}
    print(f"[{label}] {len(tasks)} fits on {min(n_jobs, len(tasks))} workers, "
          f"pooled in {time.time() - t0:.1f}s")
    return pooled, per_model


def attach_mi_diagnostics(summary, pooled, ppi_adjusted_drugs):
    """Add FMI / df / m to a summary table, taking each row from the model it
    was actually estimated in."""
    ppi_set = set(ppi_adjusted_drugs)
    recs = []
    for d in summary["drug_class"]:
        key = "with_ppi" if d in ppi_set else "no_ppi"
        row = pooled[key].loc[d]
        recs.append({"drug_class": d, "FMI": round(float(row["fmi"]), 3),
                     "MI df": round(float(row["df"]), 1),
                     "m": int(row["n_imputations"])})
    return summary.merge(pd.DataFrame(recs), on="drug_class", how="left")


def complete_case_frame(df, impute_cols=None, label=""):
    impute_cols = IMPUTE_COLS if impute_cols is None else impute_cols
    mask = df[list(impute_cols)].notna().all(axis=1)
    sub = df.loc[mask].reset_index(drop=True)
    print(f"[{label}] complete-case: {sub['patid'].nunique():,} of "
          f"{df['patid'].nunique():,} patients "
          f"({100 * sub['patid'].nunique() / df['patid'].nunique():.1f}%), "
          f"{int(sub.loc[sub['event_flag'] == 1, 'patid'].nunique()):,} events retained")
    return sub


def compare_mi_vs_cc(mi_summary, cc_summary):
    """Side-by-side MI vs complete-case HRs with the percentage change in the
    point estimate - this is the table reviewers ask for."""
    a = mi_summary[["drug_class", "HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (MI)", "CI Lower": "CI Lower (MI)",
                 "CI Upper": "CI Upper (MI)", "p": "p (MI)"})
    b = cc_summary[["drug_class", "HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (CC)", "CI Lower": "CI Lower (CC)",
                 "CI Upper": "CI Upper (CC)", "p": "p (CC)"})
    out = a.merge(b, on="drug_class", how="outer")
    out["% change in HR"] = (100 * (out["HR (CC)"] - out["HR (MI)"]) / out["HR (MI)"]).round(1)
    return out.sort_values("HR (MI)", ascending=False).reset_index(drop=True)


check_mi_versions()


In [ ]:
# ---- MICE for the Serious bleeding cohort ---------------------------------
# Imputation is cohort-specific: the outcome and follow-up time in the
# imputation model must match the model the imputed values will be used in.
major_imp_frame = build_imputation_frame(
    major_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_terms, extra_binary=("PPI_baseline",))

print(major_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

major_imputations, major_kernel = run_mice(
    major_imp_frame, IMPUTE_COLS, label="Serious bleeding", **MI_CONFIG)

major_imp_diag = imputation_diagnostics(major_imp_frame, major_imputations, IMPUTE_COLS)
major_imp_diag


### Serious bleeding - imputed and pooled estimates


In [ ]:
# ---- Adjusted models across imputations, pooled by Rubin's rules ---------
major_pooled_fits, major_per_imp_fits = fit_adjusted_models_mi(
    major_df, major_imputations, label="Serious bleeding")

# Crude HRs contain no imputed covariate, so they are fitted once on the
# observed data and need no pooling.
crude_hr_major_df = crude_hr_table(major_df, drug_terms, label="Serious bleeding")

major_summary = build_split_ppi_summary(major_pooled_fits, major_df,
                                    drug_terms, ppi_adjusted_drugs)
major_summary = attach_mi_diagnostics(major_summary, major_pooled_fits, ppi_adjusted_drugs)
major_summary = major_summary.merge(crude_hr_major_df, on="drug_class", how="left")
major_summary.to_excel(os.path.join(SENS_LAG_DIR, "hr_drug_name_major_summary.xlsx"), index=False)
major_summary


### Minor Bleeding


In [ ]:
minor_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx"), dtype=str)
minor_doac_exposure["age_at_study_start"] = minor_doac_exposure["age_at_study_start"].astype(int)
minor_doac_exposure["study_start_date"] = pd.to_datetime(minor_doac_exposure["study_start_date"])
minor_doac_exposure["study_end_date"] = pd.to_datetime(minor_doac_exposure["study_end_date"])
minor_doac_exposure["event_date"] = pd.to_datetime(minor_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

minor_doac_exposure = minor_doac_exposure[(minor_doac_exposure["age_at_study_start"] >= 18) &
                (minor_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (minor_doac_exposure["taken_warfarin_1w_before"] =="0") &
                    (minor_doac_exposure["taken_warfarin_during_study"] =="0")]

minor_doac_exposure = minor_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date"]]

In [ ]:
minor_interacting_drugs_presc = interacting_drugs_exposure.copy()
minor_interacting_drugs_presc["study_end_date"] = minor_interacting_drugs_presc["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
minor_interacting_drugs_presc = minor_interacting_drugs_presc[minor_interacting_drugs_presc["issuedate"] < minor_interacting_drugs_presc["study_end_date"]]
minor_interacting_drugs_presc.sort_values(['patid', 'issuedate'], inplace=True)
minor_interacting_drugs_presc.reset_index(drop=True, inplace=True)
minor_interacting_drugs_presc

In [ ]:
minor_interacting_drugs_presc_merged = merge_prescriptions(minor_interacting_drugs_presc)
minor_interacting_drugs_presc_merged["study_start_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_interacting_drugs_presc_merged["study_end_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

minor_interacting_drugs_presc_merged

In [ ]:
minor_exposure_timeline_df = build_exposure_timeline(minor_interacting_drugs_presc_merged)
minor_exposure_timeline_df["study_start_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_exposure_timeline_df["study_end_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
minor_exposure_timeline_df["event_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["event_date"])
minor_exposure_timeline_df["event_flag"] = np.where(
    (minor_exposure_timeline_df["event_date"] >= minor_exposure_timeline_df["start_date"]) & 
    (minor_exposure_timeline_df["event_date"] <= minor_exposure_timeline_df["end_date"]), 1, 0)
minor_exposure_timeline_df

In [ ]:
minor_events_df = minor_doac_exposure[minor_doac_exposure["event_date"].notnull()]

In [ ]:
minor_events_df = minor_doac_exposure[minor_doac_exposure["event_date"].notnull()] 

pats_with_exposures = set(minor_exposure_timeline_df["patid"].unique())

unique_minor_events_df = minor_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = minor_doac_exposure[~minor_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_minor_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

minor_exposure_timeline_df = pd.concat([minor_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
minor_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:

minor_df = minor_exposure_timeline_df.copy()
for drug in drugs_to_consider:
    minor_df[f"{drug}"] = minor_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)

for col in binary_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

for col in continuous_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col])

minor_df = minor_df[minor_df["start_date"] < minor_df["end_date"]].reset_index(drop=True)
minor_df["PPI_baseline"] = minor_df["patid"].isin(ppi_pats).astype(int)

In [ ]:
# ---- MICE for the Minor bleeding cohort ---------------------------------
# Imputation is cohort-specific: the outcome and follow-up time in the
# imputation model must match the model the imputed values will be used in.
minor_imp_frame = build_imputation_frame(
    minor_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_terms, extra_binary=("PPI_baseline",))

print(minor_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

minor_imputations, minor_kernel = run_mice(
    minor_imp_frame, IMPUTE_COLS, label="Minor bleeding", **MI_CONFIG)

minor_imp_diag = imputation_diagnostics(minor_imp_frame, minor_imputations, IMPUTE_COLS)
minor_imp_diag


### Minor bleeding - imputed and pooled estimates


In [ ]:
# ---- Adjusted models across imputations, pooled by Rubin's rules ---------
minor_pooled_fits, minor_per_imp_fits = fit_adjusted_models_mi(
    minor_df, minor_imputations, label="Minor bleeding")

# Crude HRs contain no imputed covariate, so they are fitted once on the
# observed data and need no pooling.
crude_hr_minor_df = crude_hr_table(minor_df, drug_terms, label="Minor bleeding")

minor_summary = build_split_ppi_summary(minor_pooled_fits, minor_df,
                                    drug_terms, ppi_adjusted_drugs)
minor_summary = attach_mi_diagnostics(minor_summary, minor_pooled_fits, ppi_adjusted_drugs)
minor_summary = minor_summary.merge(crude_hr_minor_df, on="drug_class", how="left")
minor_summary.to_excel(os.path.join(SENS_LAG_DIR, "hr_drug_name_minor_summary.xlsx"), index=False)
minor_summary


### All Bleedings

In [ ]:
all_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_all.xlsx"), dtype=str)
all_doac_exposure["age_at_study_start"] = all_doac_exposure["age_at_study_start"].astype(int)
all_doac_exposure["study_start_date"] = pd.to_datetime(all_doac_exposure["study_start_date"])
all_doac_exposure["study_end_date"] = pd.to_datetime(all_doac_exposure["study_end_date"])
all_doac_exposure["event_date"] = pd.to_datetime(all_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

all_doac_exposure = all_doac_exposure[(all_doac_exposure["age_at_study_start"] >= 18) &
                (all_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (all_doac_exposure["taken_warfarin_1w_before"] =="0") &
                    (all_doac_exposure["taken_warfarin_during_study"] =="0")]

all_doac_exposure = all_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date"]]

In [ ]:
all_interacting_drugs_presc = interacting_drugs_exposure.copy()
all_interacting_drugs_presc["study_end_date"] = all_interacting_drugs_presc["patid"].map(all_doac_exposure.set_index("patid")["study_end_date"])
all_interacting_drugs_presc = all_interacting_drugs_presc[all_interacting_drugs_presc["issuedate"] < all_interacting_drugs_presc["study_end_date"]]
all_interacting_drugs_presc.sort_values(['patid', 'issuedate'], inplace=True)
all_interacting_drugs_presc.reset_index(drop=True, inplace=True)
all_interacting_drugs_presc

In [ ]:
all_interacting_drugs_presc_merged = merge_prescriptions(all_interacting_drugs_presc)
all_interacting_drugs_presc_merged["study_start_date"] = all_interacting_drugs_presc_merged["patid"].map(all_doac_exposure.set_index("patid")["study_start_date"])
all_interacting_drugs_presc_merged["study_end_date"] = all_interacting_drugs_presc_merged["patid"].map(all_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

all_interacting_drugs_presc_merged

In [ ]:
all_exposure_timeline_df = build_exposure_timeline(all_interacting_drugs_presc_merged)
all_exposure_timeline_df["study_start_date"] = all_exposure_timeline_df["patid"].map(all_doac_exposure.set_index("patid")["study_start_date"])
all_exposure_timeline_df["study_end_date"] = all_exposure_timeline_df["patid"].map(all_doac_exposure.set_index("patid")["study_end_date"])
all_exposure_timeline_df["event_date"] = all_exposure_timeline_df["patid"].map(all_doac_exposure.set_index("patid")["event_date"])
all_exposure_timeline_df["event_flag"] = np.where(
    (all_exposure_timeline_df["event_date"] >= all_exposure_timeline_df["start_date"]) & 
    (all_exposure_timeline_df["event_date"] <= all_exposure_timeline_df["end_date"]), 1, 0)
all_exposure_timeline_df

In [ ]:
all_events_df = all_doac_exposure[all_doac_exposure["event_date"].notnull()] 

pats_with_exposures = set(all_exposure_timeline_df["patid"].unique())

unique_all_events_df = all_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = all_doac_exposure[~all_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_all_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

all_exposure_timeline_df = pd.concat([all_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
all_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:

all_df = all_exposure_timeline_df.copy()
for drug in drugs_to_consider:
    all_df[f"{drug}"] = all_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)

for col in binary_comorbidity_cols:
    all_df[col] = all_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

for col in continuous_comorbidity_cols:
    all_df[col] = all_df["patid"].map(comorbidities.set_index("patid")[col])

all_df = all_df[all_df["start_date"] < all_df["end_date"]].reset_index(drop=True)
all_df["PPI_baseline"] = all_df["patid"].isin(ppi_pats).astype(int)

In [ ]:
# ===========================================================================
# STEP 1: the drug-free reference stratum for this analysis
# ===========================================================================
# RD = r0 x (HR - 1), where r0 is the event rate in person-time with none of
# this model's exposure terms active. Needs only the interval frames - no Cox
# fit, no MICE.
#
# This stratum is SPECIFIC TO THIS NOTEBOOK: "no interacting drug active" means
# something different for 18 named drugs, for 8 classes, for the PK/PD groups
# and for the drug-count categories. Each analysis therefore writes its own
# file, and all_hr_combine.ipynb reads them per group.
#
# r0 also differs between outcomes, because follow-up ends at the first event of
# that type - so all three are reported.

_ref_rows = []
for _lbl, _d in (("Serious", major_df), ("Minor", minor_df), ("All", all_df)):
    _ev, _py, _rate = _events_and_py(_d, reference_mask(_d, drug_terms))
    _ref_rows.append({"outcome": _lbl, "ref_events": _ev,
                      "ref_person_years": round(_py, 2),
                      "rate_per_100py": round(_rate, 3)})

reference_strata_table = pd.DataFrame(_ref_rows)
display(reference_strata_table)

os.makedirs(SENS_LAG_DIR, exist_ok=True)
reference_strata_table.to_excel(os.path.join(SENS_LAG_DIR, "reference_strata_drug_name.xlsx"), index=False)
print("wrote results/reference_strata_drug_name.xlsx")

# Sanity check: every exposed stratum has a rate above the cohort mean, so the
# drug-free rate MUST sit below it. Fill in the rates you report in the Results.
COHORT_RATE_PER_100PY = {"Serious": 1.8}          # add "Minor" and "All"
for _r in _ref_rows:
    _c = COHORT_RATE_PER_100PY.get(_r["outcome"])
    if _c is None:
        print(f'  {_r["outcome"]}: {_r["rate_per_100py"]:.3f} per 100 PY (not checked - '
              'no cohort rate given)')
    else:
        _ok = _r["rate_per_100py"] < _c
        print(f'  {_r["outcome"]}: {_r["rate_per_100py"]:.3f} vs cohort {_c} -> '
              f'{"OK" if _ok else "FAILED - reference mask not applied, do not use"}')


In [ ]:
# ---- MICE for the All bleeding cohort ---------------------------------
# Imputation is cohort-specific: the outcome and follow-up time in the
# imputation model must match the model the imputed values will be used in.
all_imp_frame = build_imputation_frame(
    all_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_terms, extra_binary=("PPI_baseline",))

print(all_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

all_imputations, all_kernel = run_mice(
    all_imp_frame, IMPUTE_COLS, label="All bleeding", **MI_CONFIG)

all_imp_diag = imputation_diagnostics(all_imp_frame, all_imputations, IMPUTE_COLS)
all_imp_diag


### All bleeding - imputed and pooled estimates


In [ ]:
# ---- Adjusted models across imputations, pooled by Rubin's rules ---------
all_pooled_fits, all_per_imp_fits = fit_adjusted_models_mi(
    all_df, all_imputations, label="All bleeding")

# Crude HRs contain no imputed covariate, so they are fitted once on the
# observed data and need no pooling.
crude_hr_all_df = crude_hr_table(all_df, drug_terms, label="All bleeding")

all_summary = build_split_ppi_summary(all_pooled_fits, all_df,
                                    drug_terms, ppi_adjusted_drugs)
all_summary = attach_mi_diagnostics(all_summary, all_pooled_fits, ppi_adjusted_drugs)
all_summary = all_summary.merge(crude_hr_all_df, on="drug_class", how="left")
all_summary.to_excel(os.path.join(SENS_LAG_DIR, "hr_drug_name_all_summary.xlsx"), index=False)
all_summary

In [ ]:
columns_mapping = {
    "Patient Counts": "patient_counts",
    "HR": "hr",
    "CI Lower": "ci_lower",
    "CI Upper": "ci_upper",
    "p": "p",
    "No of Events": "no_of_events",
    "Crude HR":"crude_hr",
    "Crude CI Lower":"crude_ci_lower",
    "Crude CI Upper":"crude_ci_upper",
    "Crude p":"crude_p",
}


major_summary.rename(columns=columns_mapping, inplace=True)
minor_summary.rename(columns=columns_mapping, inplace=True)

In [ ]:
# Final table columns. The MI diagnostics (FMI, Barnard-Rubin df, m) are kept
# if present so they can go straight into the supplementary table.
extra_mi_cols = [c for c in ["FMI", "MI df", "m"] if c in major_summary.columns]

columns_to_keep = ["drug_class", "PPI adjusted", "patient_counts", "events/person_years",
                   "events_per_100py", "hr", "ci_lower", "ci_upper", "p",
                   "crude_hr", "crude_ci_lower", "crude_ci_upper", "crude_p",
                   "NNH", "NNH_CI_lower", "NNH_CI_upper"] + extra_mi_cols
major_summary = major_summary[columns_to_keep]
minor_summary = minor_summary[columns_to_keep]

columns_to_rename = {
    "drug_class": "Drug Name",
    "patient_counts": "Patient (n)",
    "events/person_years": "Events/Person Years",
    "events_per_100py": "Events/100 Person Years",
    "hr": "Adjusted HR",
    "ci_lower": "Adjusted HR CI Lower",
    "ci_upper": "Adjusted HR CI Upper",
    "p": "Adjusted HR p-value",
    "crude_hr": "Crude HR",
    "crude_ci_lower": "Crude HR CI Lower",
    "crude_ci_upper": "Crude HR CI Upper",
    "crude_p": "Crude HR p-value",
    "NNH": "NNH (per 1000 PY)",
    "NNH_CI_lower": "NNH CI Lower",
    "NNH_CI_upper": "NNH CI Upper",
    "FMI": "Fraction of missing information",
    "MI df": "MI degrees of freedom",
    "m": "Imputations (m)",
}

major_summary = major_summary.rename(columns=columns_to_rename)
minor_summary = minor_summary.rename(columns=columns_to_rename)


In [ ]:
major_summary["type"] = "Severe"
minor_summary["type"] = "Minor"

major_minor_summary = pd.concat(
    [major_summary, minor_summary],
    ignore_index=True
)

major_minor_summary["type"] = pd.Categorical(
    major_minor_summary["type"],
    categories=["Severe", "Minor"],
    ordered=True
)

# Maximum HR per drug
drug_order = (
    major_minor_summary.groupby("Drug Name")["Adjusted HR"]
    .max()
    .sort_values(ascending=False)
)

major_minor_summary["drug_rank"] = (
    major_minor_summary["Drug Name"]
    .map({drug: i for i, drug in enumerate(drug_order.index)})
)

major_minor_summary = (
    major_minor_summary
    .sort_values(["drug_rank", "type"])
    .drop(columns="drug_rank")
    .reset_index(drop=True)
)
columns_to_keep = ["Drug Name", "PPI adjusted", "type", "Patient (n)", "Events/Person Years", "Events/100 Person Years", "Adjusted HR", "Adjusted HR CI Lower", "Adjusted HR CI Upper", "Adjusted HR p-value", "NNH (per 1000 PY)", "NNH CI Lower", "NNH CI Upper"]
major_minor_summary = major_minor_summary[columns_to_keep]
major_minor_summary

## Sensitivity analysis: lagged exposure

The primary analysis classifies person-time as exposed from the day an interacting
drug is issued. To assess protopathic bias, the exposure-classification pipeline is
re-run with the onset of exposure lagged by L days, applying the same lag to every
drug (`lag_sensitivity.py`). Table S17 reports the 3-day lag; `LAG_DAYS_TO_RUN`
also includes lag 0, which reproduces the primary analysis through the same rebuild
path and serves as a self-check.

The lag is applied *after* `merge_prescriptions`, so the 7-day (antibiotics,
antifungals) and 14-day (all other classes) permissible gaps are unchanged. In the
default `delay_onset` mode an exposure episode `[s, e)` becomes `[s + L, e)`;
person-time no longer exposed returns to the drug-free reference stratum, and the
cohort, follow-up, events and censoring are unchanged. `exposure_retention()`
reports the exposed events and person-years retained at each lag.

In [ ]:
# ===========================================================================
# Exposure-lag sensitivity analyses - setup
# ===========================================================================
# lag_sensitivity.py must sit next to this notebook. It contains no modelling
# code: the notebook's own fitters and table builders are handed to it, so the
# lagged analyses go through exactly the same code path as the primary one.
import sys, os, importlib
if "" not in sys.path:
    sys.path.insert(0, "")
import lag_sensitivity as lagmod
importlib.reload(lagmod)                     # picks up edits without a restart
from lag_sensitivity import (lag_episodes, build_lagged_frame, exposure_retention,
                             run_lag_sensitivity, combine_lag_summaries,
                             lag_comparison_wide)

# ---- Configuration --------------------------------------------------------
LAG_DAYS_TO_RUN = [0, 3, 7, 14]   # 0 = primary analysis, kept as the self-check
LAG_MODE        = "delay_onset"   # or "shift_window" (see the markdown above)
LAG_INCLUDE_CRUDE = False         # crude HRs add one univariate fit per term per
                                  # lag and are not needed for the supplement;
                                  # set True if a reviewer asks for them
LAG_RESULTS_DIR = SENS_LAG_DIR

# ---- Work out which notebook this is, without editing anything ------------
# hr_drugs.ipynb models individual drugs (episodes keyed on 'drug_name');
# hr_drug_class.ipynb models classes (episodes keyed on 'drug_class').
LAG_TERM_COL = ("drug_name"
                if "drug_name" in major_interacting_drugs_presc_merged.columns
                else "drug_class")
LAG_FILE_STEM = ("lag_sensitivity_drug_name" if LAG_TERM_COL == "drug_name"
                 else "lag_sensitivity_drug_class")
LAG_PPI_ADJUSTED = (PPI_ADJUSTED_CLASSES if "PPI_ADJUSTED_CLASSES" in globals()
                    else ppi_adjusted_drugs)
LAG_BACKFILL_AGE = backfill_age if "backfill_age" in globals() else None

# The notebook's own functions - no duplicated modelling logic.
LAG_MODEL_FNS = dict(
    fit_mi      = fit_adjusted_models_mi,     # (df, imputations, label=) -> (pooled, per_imp)
    summary     = build_split_ppi_summary,    # (pooled, df, terms, ppi_adjusted)
    attach_diag = attach_mi_diagnostics,      # (summary, pooled, ppi_adjusted)
    crude       = crude_hr_table,             # (df, terms, label=)
)

# One entry per outcome cohort. The merged prescription episodes and the cohort
# frame are outcome-specific because follow-up ends at the first event of that
# type, so each outcome must be lagged and rebuilt on its own episodes.
LAG_OUTCOMES = {
    "Serious": dict(presc_merged=major_interacting_drugs_presc_merged,
                    doac_exposure=major_doac_exposure,
                    imputations=major_imputations),
    "Minor":   dict(presc_merged=minor_interacting_drugs_presc_merged,
                    doac_exposure=minor_doac_exposure,
                    imputations=minor_imputations),
    "All":     dict(presc_merged=all_interacting_drugs_presc_merged,
                    doac_exposure=all_doac_exposure,
                    imputations=all_imputations),
}

print(f"exposure term column : {LAG_TERM_COL}")
print(f"terms ({len(drug_terms)})          : {', '.join(map(str, drug_terms))}")
print(f"PPI-adjusted terms   : {LAG_PPI_ADJUSTED}")
print(f"lags                 : {LAG_DAYS_TO_RUN} ({LAG_MODE})")
print(f"outcomes             : {list(LAG_OUTCOMES)}")
print(f"output stem          : {LAG_RESULTS_DIR}/{LAG_FILE_STEM}_*.xlsx")

# NOTE ON IMPUTATION. The imputations built for the primary analysis are reused
# for every lag. That is appropriate because lagging changes only the LABELLING
# of person-time as exposed or unexposed: the cohort, the follow-up time, the
# events and the imputed variables (eGFR, BMI, both measured at baseline) are
# all unchanged. The ever-exposed flags inside the imputation model shift
# slightly, but they are predictors of baseline covariates, not of the outcome
# model. State this in the supplement. If a reviewer wants MICE repeated inside
# each lag, rebuild the frame first with build_lagged_frame(), then call
# build_imputation_frame() / run_mice() on it and pass the result through.

In [ ]:
# ===========================================================================
# Exposure-lag sensitivity analyses - run
# ===========================================================================
# Runtime: one full rebuild + (2 x m) pooled Cox fits per lag per outcome, so
# roughly (number of lags) x the cost of the primary analysis. Outcomes are run
# sequentially and the interval frames are released as soon as the models are
# fitted (keep_frames=False), so peak memory is about that of one outcome.
#
# To run a subset while testing, cut LAG_OUTCOMES or LAG_DAYS_TO_RUN, e.g.
#     _outcomes = {"Serious": LAG_OUTCOMES["Serious"]}
#     LAG_DAYS_TO_RUN = [0, 7]

lag_summaries   = {}     # {outcome: {lag: summary DataFrame}}
lag_retention   = {}     # {outcome: retention DataFrame}

for _outcome, _cfg in LAG_OUTCOMES.items():
    print("=" * 78)
    print(f"LAG SENSITIVITY - {_outcome} bleeding")
    print("=" * 78)
    _summ, _ret, _ = run_lag_sensitivity(
        presc_merged   = _cfg["presc_merged"],
        doac_exposure  = _cfg["doac_exposure"],
        imputations    = _cfg["imputations"],
        terms          = drug_terms,
        term_col       = LAG_TERM_COL,
        ppi_adjusted   = LAG_PPI_ADJUSTED,
        outcome        = _outcome,
        model_fns      = LAG_MODEL_FNS,
        timeline_fn    = build_exposure_timeline,
        comorbidities  = comorbidities,
        binary_cols    = binary_comorbidity_cols,
        continuous_cols= continuous_comorbidity_cols,
        ppi_pats       = ppi_pats,
        lags           = LAG_DAYS_TO_RUN,
        mode           = LAG_MODE,
        backfill_age   = LAG_BACKFILL_AGE,
        include_crude  = LAG_INCLUDE_CRUDE,
        keep_frames    = False,
        results_dir    = LAG_RESULTS_DIR,
        file_stem      = LAG_FILE_STEM,
    )
    lag_summaries[_outcome] = _summ
    lag_retention[_outcome] = _ret

lag_retention_all = pd.concat(lag_retention.values(), ignore_index=True)
lag_retention_all.to_excel(f"{LAG_RESULTS_DIR}/{LAG_FILE_STEM}_retention_all.xlsx",
                           index=False)
lag_retention_all.head(20)

In [ ]:
# ===========================================================================
# Self-check: lag 0 must reproduce the primary analysis
# ===========================================================================
# If this prints a non-trivial difference, the rebuild path in
# build_lagged_frame() does not match the primary pipeline and the lagged
# estimates below must not be reported. Tolerance is 0.01 on the hazard ratio -
# the primary tables are rounded to 2 dp, so an exact match is not expected.

def check_lag0_matches_primary(lag0_summary, primary_summary, outcome, tol=0.011):
    a = lag0_summary[["drug_class", "HR", "CI Lower", "CI Upper"]].copy()
    b = primary_summary[["drug_class", "HR", "CI Lower", "CI Upper"]].copy()
    m = a.merge(b, on="drug_class", how="outer", suffixes=(" lag0", " primary"))
    m["abs diff"] = (m["HR lag0"] - m["HR primary"]).abs().round(4)
    bad = m[(m["abs diff"] > tol) | m["abs diff"].isna()]
    status = "OK" if bad.empty else "MISMATCH - investigate before reporting"
    print(f"[{outcome}] lag-0 vs primary: {status} "
          f"(max |diff| = {m['abs diff'].max()})")
    return m.sort_values("abs diff", ascending=False)

_primary = {"Serious": major_summary, "Minor": minor_summary, "All": all_summary}
lag0_checks = {}
for _o, _p in _primary.items():
    if _o in lag_summaries and 0 in lag_summaries[_o]:
        # major_summary etc. may already have been renamed by the reporting
        # cells further down; fall back to the saved primary workbook if so.
        _pp = _p if "HR" in _p.columns else None
        if _pp is None:
            print(f"[{_o}] primary table already renamed - reload it from "
                  f"results/ if you need this check.")
            continue
        lag0_checks[_o] = check_lag0_matches_primary(lag_summaries[_o][0], _pp, _o)

lag0_checks.get("Serious")

In [ ]:
# ===========================================================================
# Supplementary tables: lagged estimates side by side with the primary
# ===========================================================================
lag_long = combine_lag_summaries(lag_summaries)
lag_long.to_excel(f"{LAG_RESULTS_DIR}/{LAG_FILE_STEM}_long.xlsx", index=False)

# Wide layout, one block per lag, with the exposed events each estimate rests on
lag_wide = pd.concat(
    [lag_comparison_wide(lag_long, outcome=o, retention=lag_retention[o])
     for o in lag_summaries],
    ignore_index=True)
lag_wide.to_excel(f"{LAG_RESULTS_DIR}/{LAG_FILE_STEM}_wide.xlsx", index=False)

print(f"wrote {LAG_RESULTS_DIR}/{LAG_FILE_STEM}_long.xlsx and _wide.xlsx")
lag_wide

In [ ]:
# ===========================================================================
# Supplementary figure: hazard ratio against lag length
# ===========================================================================
# One panel per outcome, one line per drug/class. A flat line means the estimate
# does not depend on where exposure is deemed to start. A fall with increasing
# lag is consistent with reverse causation or with the exposure window being
# emptied of events - the retention table tells you which. A rise is consistent
# with genuine latency.

_outcomes = list(lag_summaries)
fig, axes = plt.subplots(1, len(_outcomes), figsize=(6 * len(_outcomes), 7),
                         sharey=True, squeeze=False)

for ax, outcome in zip(axes[0], _outcomes):
    sub = lag_long[lag_long["outcome"] == outcome]
    order = (sub[sub["lag_days"] == 0].sort_values("HR", ascending=False)["drug_class"]
             .tolist())
    ret = lag_retention[outcome].set_index(["lag_days", "term"])
    for term in order:
        s = sub[sub["drug_class"] == term].sort_values("lag_days")
        # hollow markers where the estimate rests on <10 exposed events
        few = [bool(ret.loc[(l, term), "below_event_threshold"])
               if (l, term) in ret.index else False for l in s["lag_days"]]
        ax.plot(s["lag_days"], s["HR"], marker="o", lw=1.4, ms=6, label=term,
                markerfacecolor="none" if all(few) else None)
        for x, y, f in zip(s["lag_days"], s["HR"], few):
            if f:
                ax.plot(x, y, marker="o", ms=6, mfc="white",
                        mec=ax.lines[-1].get_color())
    ax.axhline(1.0, color="0.4", lw=1, ls="--")
    ax.set_xticks(sorted(sub["lag_days"].unique()))
    ax.set_xlabel("Exposure lag (days)")
    ax.set_title(f"{outcome} bleeding")
    ax.set_yscale("log")
    ax.set_yticks([0.5, 0.75, 1, 1.5, 2, 3, 4])
    ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

axes[0][0].set_ylabel("Adjusted hazard ratio (log scale)")
axes[0][-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8,
                   frameon=False, title="hollow marker = <10 exposed events")
fig.tight_layout()
#            bbox_inches="tight")
print(f"wrote {LAG_RESULTS_DIR}/{LAG_FILE_STEM}_figure.png")
plt.show()